# 🇳🇬 Naija Persona Agent — Corpus Build via OpenAI (Alternative to 01)

**Same output as `01_build_corpus.ipynb` — different backend.** Use this notebook if NVIDIA NIM rate limits are blocking you and you have OpenAI API budget.

**Run on CPU runtime.** No GPU credits needed.

## Why OpenAI?
- **Much higher rate limits** — tier 1 OpenAI handles 20+ concurrent easily; tier 2+ handles 100+. Free NVIDIA NIM hits 429s at concurrency 6.
- **GPT-4o is excellent at Nigerian Pidgin** — trained on substantially more Nigerian-language web data than European-focused models.
- **Fast end-to-end** — 3k corpus takes ~10-15 min on OpenAI vs 30-40 min on rate-limited NIM.
- **Predictable cost** — see table below.

## Cost estimates (USD)

| Corpus size | gpt-4o-mini (fast/cheap) | gpt-4o (best Pidgin quality) |
|---|---|---|
| 3,000 records | ~$5 | ~$25 |
| 10,000 records | ~$15 | ~$80 |
| 20,000 records | ~$30 | ~$160 |

**For hackathon submission**: use gpt-4o for the headline Pidgin quality. Cost is rounding error vs the value of a defensible cultural-fidelity claim.

## Inputs
- `OPENAI_API_KEY` — set in Colab Secrets

## Outputs (separate Drive root from 01_build_corpus.ipynb)

Stored at `/content/drive/MyDrive/naija_persona_corpus_openai/` so you can keep both
corpora side-by-side and compare. To train on this corpus, set `CORPUS_SOURCE="openai"`
in 02_finetune.ipynb (default).

- `processed/v1_train_alpaca.jsonl`
- `processed/v1_val_alpaca.jsonl`
- `processed/v1_test_alpaca.jsonl`
- `processed/final_alpaca_format.jsonl` + ShareGPT + Parquet
- `augmented/checkpoints/pipeline{1,2,3}/batch_*.parquet`
- `metadata/final_statistics.json`

## 🌐 Backend options — freemodel / OpenAI / LM Studio / Ollama

Set `LLM_BACKEND` in the cell below to switch backends.

### freemodel.dev (default) — GPT-5.5 via Responses API
User's preferred backend. Higher concurrency than OpenAI free tier.
- Set `OPENAI_API_KEY` in Colab Secrets to your freemodel key
- `LLM_BACKEND = "freemodel"` (default)
- `PRIMARY_MODEL = "gpt-5.5"` (auto-set)
- `WIRE_API = "responses"` (auto-set; uses /v1/responses not /v1/chat/completions)

### Local backends (LM Studio / Ollama) — free, $0
Set `LLM_BACKEND = "lm_studio"` (or `"ollama"`) to run **fully local, zero cost**.

### LM Studio quick setup
1. Download LM Studio → https://lmstudio.ai
2. Search + download a model (recommended starters):
   - **Llama 3.1 8B Instruct** (`bartowski/Meta-Llama-3.1-8B-Instruct-GGUF`, Q4_K_M) — 5GB, fits anywhere, decent Pidgin
   - **Llama 3.3 70B Instruct** (`bartowski/Llama-3.3-70B-Instruct-GGUF`, Q4_K_M) — 40GB, needs 64GB+ RAM, best Pidgin
   - **Nemotron Super 49B** (`bartowski/Llama-3.3-Nemotron-Super-49B-v1-GGUF`, Q5_K_M) — 33GB, RLHF, very good
3. **Developer → Start Server** (port 1234)
4. Set `LLM_BACKEND = "lm_studio"` below and set `PRIMARY_MODEL` to the exact name shown in LM Studio.

### Ollama quick setup (alternative)
```bash
brew install ollama  # or download from ollama.ai
ollama serve
ollama pull llama3.1:8b-instruct        # ~5GB, decent
# or:
ollama pull llama3.3:70b-instruct       # ~40GB, best quality
```
Then `LLM_BACKEND = "ollama"` and `PRIMARY_MODEL = "llama3.1:8b-instruct"`.

### Colab → local backend
If you're running this notebook in Colab but want to use LM Studio on your laptop, expose the local port:
```bash
ngrok http 1234     # gives you https://abc123.ngrok.io
```
Then set `LM_STUDIO_URL = "https://abc123.ngrok.io/v1"` in the cell below.
Most users will just run this notebook locally (Jupyter on their machine, not Colab) for the local backend.

## 0. Setup & Install

In [ ]:
# Minimal install — no data_designer, no Unsloth conflicts
!pip install -q openai==1.55.* jsonlines pandas tqdm nest_asyncio httpx "pyarrow>=19.0.1,<20" datasets
print("✅ Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.6/389.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 17.9 MB/s eta 0:00:00
✅ Dependencies installed.


In [ ]:
# IMPORTS
import os, sys, json, time, csv, asyncio, traceback, random, re
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import pandas as pd
import jsonlines
import httpx
import nest_asyncio; nest_asyncio.apply()
from tqdm.notebook import tqdm

from openai import AsyncOpenAI

IN_COLAB = "google.colab" in sys.modules
print("✅ Imports loaded.")

✅ Imports loaded.


In [ ]:
# MOUNT GOOGLE DRIVE — same path as 01_build_corpus.ipynb
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE = Path("/content/drive/MyDrive/naija_persona_corpus_openai")
else:
    DRIVE_BASE = Path.home() / "naija_persona_corpus_openai"
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR    = DRIVE_BASE
RAW_DIR       = OUTPUT_DIR / "raw"
PROCESSED_DIR = OUTPUT_DIR / "processed"
METADATA_DIR  = OUTPUT_DIR / "metadata"
AUGMENTED_DIR = OUTPUT_DIR / "augmented"
CHECKPOINT_DIR = AUGMENTED_DIR / "checkpoints"
P1_CKPT_DIR    = CHECKPOINT_DIR / "pipeline1"
P2_CKPT_DIR    = CHECKPOINT_DIR / "pipeline2"
P3_CKPT_DIR    = CHECKPOINT_DIR / "pipeline3"
for d in [RAW_DIR, PROCESSED_DIR, METADATA_DIR, AUGMENTED_DIR,
          CHECKPOINT_DIR, P1_CKPT_DIR, P2_CKPT_DIR, P3_CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f"✅ Drive base: {DRIVE_BASE}")

Mounted at /content/drive
✅ Drive base: /content/drive/MyDrive/naija_persona_corpus_openai


In [ ]:
# BACKEND SELECTOR — choose where the LLM calls go
# Options:
#   "freemodel"   → freemodel.dev (GPT-5.5 via OpenAI Responses API; user's preferred path)
#   "openai"      → official OpenAI cloud API (Chat Completions)
#   "lm_studio"   → LM Studio local server (free, runs on your laptop)
#   "ollama"      → Ollama local server (free, alternative to LM Studio)
LLM_BACKEND = "freemodel"

LM_STUDIO_URL = "http://localhost:1234/v1"
OLLAMA_URL    = "http://localhost:11434/v1"
FREEMODEL_URL = "https://api.freemodel.dev/v1"

OPENAI_BASE_URL = None
WIRE_API = "chat"   # 'chat' = chat.completions API; 'responses' = OpenAI Responses API

if LLM_BACKEND == "freemodel":
    OPENAI_BASE_URL = FREEMODEL_URL
    WIRE_API = "responses"   # freemodel.dev uses the Responses API
    if IN_COLAB:
        try:
            from google.colab import userdata
            # Reuse OPENAI_API_KEY slot; user puts their freemodel key there
            os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
            print("✅ freemodel key loaded from Colab Secrets (OPENAI_API_KEY)")
        except Exception:
            pass
    if not os.environ.get("OPENAI_API_KEY"):
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("freemodel API key: ")
    print(f"🌐 Backend: freemodel @ {FREEMODEL_URL} (Responses API)")

elif LLM_BACKEND == "lm_studio":
    OPENAI_BASE_URL = LM_STUDIO_URL
    os.environ["OPENAI_API_KEY"] = "lm-studio"
    print(f"🖥️  Backend: LM Studio @ {LM_STUDIO_URL}")

elif LLM_BACKEND == "ollama":
    OPENAI_BASE_URL = OLLAMA_URL
    os.environ["OPENAI_API_KEY"] = "ollama"
    print(f"🖥️  Backend: Ollama @ {OLLAMA_URL}")

else:  # openai
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        except Exception:
            pass
    if not os.environ.get("OPENAI_API_KEY"):
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (sk-…): ")
    print("☁️  Backend: OpenAI cloud (Chat Completions)")

assert os.environ.get("OPENAI_API_KEY"), "API key required for the chosen backend"

✅ freemodel key loaded from Colab Secrets (OPENAI_API_KEY)
🌐 Backend: freemodel @ https://api.freemodel.dev/v1 (Responses API)


## 1. Settings

In [ ]:
# GENERATION SETTINGS
NUM_P1_RECORDS = 12000   # seed-grounded from Jumia reviews
NUM_P2_RECORDS = 5000   # pure synthetic
BATCH_SIZE     = 500    # save to Drive after each batch
JUDGE_SAMPLE   = 200    # records to quality-score
CONCURRENCY    = 20     # parallel OpenAI requests; bump to 50+ if you're on tier 3+

# Models — defaults adjust based on LLM_BACKEND set above.
#
# === LLM_BACKEND="freemodel" ===
#   model="gpt-5.5"  — uses freemodel.dev's hosted GPT-5.5 (Responses API)
#   Can go higher on concurrency than free OpenAI tier.
#
# === LLM_BACKEND="openai" ===
#   gpt-4o          — best Pidgin quality (~$25 for 3k corpus)
#   gpt-4o-mini     — ~5× cheaper, slightly weaker Pidgin (~$5)
#
# === LLM_BACKEND="lm_studio" or "ollama" (local) ===
#   meta-llama-3.1-8b-instruct   — runs on 8GB+, decent Pidgin
#   meta-llama-3.3-70b-instruct  — 40GB+, best Pidgin
#   llama-3.3-nemotron-super-49b-v1  — 30GB+, RLHF, very good
if LLM_BACKEND == "freemodel":
    PRIMARY_MODEL = "gpt-5.5"
    JUDGE_MODEL   = "gpt-5.5"
    CONCURRENCY   = 40   # higher tier; bump to 60+ if no rate-limit hits
elif LLM_BACKEND == "openai":
    PRIMARY_MODEL = "gpt-4o"
    JUDGE_MODEL   = "gpt-4o"
    CONCURRENCY   = 20
else:  # local: lm_studio or ollama
    PRIMARY_MODEL = "meta-llama-3.1-8b-instruct"
    JUDGE_MODEL   = "meta-llama-3.1-8b-instruct"
    CONCURRENCY   = 4    # single local serving, low concurrency

## 2. OpenAI Async Client + Rate-Limit Backoff

In [ ]:
# Async OpenAI client with exponential backoff + dual wire-API support
_client_kwargs = {"api_key": os.environ["OPENAI_API_KEY"], "timeout": 120.0, "max_retries": 0}
if OPENAI_BASE_URL:
    _client_kwargs["base_url"] = OPENAI_BASE_URL
client = AsyncOpenAI(**_client_kwargs)

async def call_openai(
    model: str,
    system: str,
    user: str,
    max_tokens: int = 500,
    temperature: float = 0.75,
    response_format=None,
    max_retries: int = 5,
) -> str:
    """One LLM call with retry on transient errors. Honors WIRE_API."""
    last_err = None
    use_max_completion_tokens = False
    drop_temperature = False

    for attempt in range(max_retries):
        try:
            if WIRE_API == "responses":
                # OpenAI Responses API (used by freemodel.dev) requires custom raw HTTP call
                async with httpx.AsyncClient(timeout=120.0) as hc:
                    headers = {
                        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
                        "Content-Type": "application/json"
                    }
                    payload = {
                        "model": model,
                        "instructions": system,
                        "input": user,
                        "max_output_tokens": max_tokens,
                        "temperature": temperature
                    }
                    # If URL ends in /v1, we append /responses
                    base = OPENAI_BASE_URL.rstrip('/')
                    url = f"{base}/responses"
                    resp = await hc.post(url, headers=headers, json=payload)
                    resp.raise_for_status()
                    data = resp.json()

                    # freemodel typically returns output[0].content[0].text
                    try:
                        text = data["output"][0]["content"][0]["text"]
                    except (KeyError, IndexError):
                        text = str(data)
                    return text
            else:
                # Standard Chat Completions API
                kwargs = dict(
                    model=model,
                    messages=[
                        {"role": "system", "content": system},
                        {"role": "user",   "content": user},
                    ],
                )

                if use_max_completion_tokens:
                    kwargs["max_completion_tokens"] = max_tokens
                else:
                    kwargs["max_tokens"] = max_tokens

                if not drop_temperature:
                    kwargs["temperature"] = temperature

                if response_format:
                    kwargs["response_format"] = response_format

                resp = await client.chat.completions.create(**kwargs)
                return resp.choices[0].message.content
        except Exception as e:
            last_err = e
            msg = str(e).lower()

            # Auto-fallback for newer models (like o1) that reject max_tokens/temperature
            if "unsupported parameter: 'max_tokens'" in msg or "max_completion_tokens" in msg:
                use_max_completion_tokens = True
                continue
            if "unsupported parameter: 'temperature'" in msg or "'temperature' does not support" in msg:
                drop_temperature = True
                continue

            if "rate" in msg or "429" in msg or "500" in msg or "503" in msg or "timeout" in msg or "connection" in msg:
                if attempt < max_retries - 1:
                    sleep_s = 4 * (2 ** attempt) + random.uniform(0, 1)
                    await asyncio.sleep(sleep_s)
                    continue
            raise
    raise last_err

# Smoke test
async def _smoke():
    r = await call_openai(PRIMARY_MODEL, "You are helpful.", "Say 'OK'.", max_tokens=10, temperature=0)
    return (r or "").strip()
result = asyncio.run(_smoke())
print(f"✅ {LLM_BACKEND} reachable via {WIRE_API} API: {result[:60]}")

✅ openai reachable via chat API: OK


## 3. Batched Generation Helper — Auto-Save + Resume

In [ ]:
def count_existing_checkpoints(ckpt_dir: Path):
    files = sorted(ckpt_dir.glob("batch_*.parquet"))
    total = 0
    for f in files:
        try: total += len(pd.read_parquet(f))
        except: pass
    return len(files), total, files

def load_all_checkpoints(ckpt_dir: Path) -> pd.DataFrame:
    files = sorted(ckpt_dir.glob("batch_*.parquet"))
    if not files: return pd.DataFrame()
    dfs = []
    for f in files:
        try: dfs.append(pd.read_parquet(f))
        except Exception as e: print(f"  ⚠️ skipping {f.name}: {e}")
    out = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    print(f"  📦 loaded {len(out):,} from {len(dfs)} checkpoints")
    return out

async def generate_batched(
    *, generate_one_fn, total_records: int, batch_size: int,
    ckpt_dir: Path, dataset_name: str, concurrency: int = CONCURRENCY,
):
    """
    generate_one_fn: async fn that takes an index and returns a dict row.
    Runs `concurrency` parallel calls. Saves every `batch_size` rows to Drive.
    """
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    done_batches, done_records, _ = count_existing_checkpoints(ckpt_dir)

    if done_records >= total_records:
        print(f"✅ {dataset_name}: complete ({done_records:,} on Drive)")
        return load_all_checkpoints(ckpt_dir)

    remaining = total_records - done_records
    if done_records > 0:
        print(f"🔄 {dataset_name}: resuming from batch {done_batches+1} ({done_records:,} done, {remaining:,} to go)")
    else:
        print(f"🚀 {dataset_name}: starting fresh ({total_records:,} target, concurrency={concurrency})")

    batch_num = done_batches + 1
    generated = done_records
    t_start = time.time()
    sem = asyncio.Semaphore(concurrency)

    async def _wrap(idx):
        async with sem:
            try:
                return await generate_one_fn(idx)
            except Exception as e:
                return {"_error": f"{type(e).__name__}: {str(e)[:200]}", "_idx": idx}

    while generated < total_records:
        this_n = min(batch_size, total_records - generated)
        t_batch = time.time()
        print(f"\n📦 batch {batch_num} — {this_n:,} records ({100*generated/total_records:.1f}% done)")

        tasks = [_wrap(generated + i) for i in range(this_n)]
        results = []
        async for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f"batch {batch_num}") if False else _aiter_completed(tasks):
            results.append(await fut)

        # Filter errors
        ok = [r for r in results if r and "_error" not in r]
        errs = [r for r in results if r and "_error" in r]
        if errs:
            print(f"  ⚠️ {len(errs)} errors in this batch — keeping the {len(ok)} good ones")
            for e in errs[:3]:
                print(f"      • {e.get('_error', 'unknown')[:120]}")

        if not ok:
            print("  ⚠️ entire batch failed; retrying in 30s")
            await asyncio.sleep(30)
            continue

        df = pd.DataFrame(ok)
        ckpt_pq = ckpt_dir / f"batch_{batch_num:03d}.parquet"
        ckpt_jl = ckpt_dir / f"batch_{batch_num:03d}.jsonl"
        df.to_parquet(ckpt_pq, index=False)
        with jsonlines.open(ckpt_jl, mode='w') as w:
            for _, row in df.iterrows():
                w.write(row.to_dict())

        elapsed = time.time() - t_batch
        generated += len(df)
        rate = len(df) / elapsed * 60 if elapsed else 0
        print(f"  ✅ {len(df):,} records in {elapsed:.0f}s ({rate:.0f}/min) → {ckpt_pq.name}")

        # ETA
        sess_done = generated - done_records
        if sess_done > 0:
            eta = (total_records - generated) * (time.time() - t_start) / sess_done / 60
            print(f"  ⏱️ ETA: ~{eta:.0f} min")
        batch_num += 1

    print(f"\n✅ session in {(time.time()-t_start)/60:.1f}min")
    return load_all_checkpoints(ckpt_dir)

async def _aiter_completed(coros):
    """async-iter wrapper for asyncio.as_completed."""
    for fut in asyncio.as_completed(coros):
        yield fut

print("✅ batched-generation helpers ready")

✅ batched-generation helpers ready


## 4. Prepare Seed Data on Drive

In [ ]:
JUMIA_REVIEWS_URL = (
    "https://raw.githubusercontent.com/"
    "aymane-maghouti/Sentiment-Analysis-for-Jumia-Reviews-and-Smartphone-Price-Prediction-System/"
    "main/Main/Sentiment_Analysis_for_Jumia_Reviews/final_data.csv"
)
SEED_REVIEWS_PATH  = RAW_DIR / "jumia_reviews_seed.parquet"
SEED_PRODUCTS_PATH = RAW_DIR / "jumia_products_seed.parquet"

# --- Real Jumia reviews ---
if SEED_REVIEWS_PATH.exists():
    df_jumia_reviews = pd.read_parquet(SEED_REVIEWS_PATH)
    print(f"✅ Jumia reviews seed: {len(df_jumia_reviews):,}")
else:
    print("📥 downloading Jumia CSV (~15MB)...")
    raw_csv = RAW_DIR / "jumia_reviews_raw.csv"
    with httpx.Client(timeout=120.0) as cli:
        r = cli.get(JUMIA_REVIEWS_URL); r.raise_for_status()
        raw_csv.write_bytes(r.content)
    rows = []
    with raw_csv.open(encoding="utf-8", errors="replace") as fh:
        rd = csv.reader(fh); next(rd, None)
        for r in rd:
            if len(r) < 2: continue
            text = " ".join(r[0].split()).strip()
            if not (50 <= len(text) <= 1500): continue
            try: tgt = int(r[1])
            except: continue
            rows.append({"review_text": text, "binary_sentiment": tgt})
    df_jumia_reviews = pd.DataFrame(rows)
    df_jumia_reviews.to_parquet(SEED_REVIEWS_PATH, index=False)
    print(f"✅ saved {len(df_jumia_reviews):,} reviews")

# --- Product catalog ---
if SEED_PRODUCTS_PATH.exists():
    df_jumia_products = pd.read_parquet(SEED_PRODUCTS_PATH)
    print(f"✅ Jumia products: {len(df_jumia_products):,}")
else:
    print("📥 downloading Idowenst product catalog from HF...")
    from datasets import load_dataset
    ds = load_dataset("Idowenst/jumia_dataset", split="train")
    df_jumia_products = pd.DataFrame([
        {"title": r.get("name",""), "category": r.get("category",""),
         "description": (r.get("description") or "")[:600], "price_naira": r.get("price")}
        for r in ds if r.get("name")
    ])
    df_jumia_products.to_parquet(SEED_PRODUCTS_PATH, index=False)
    print(f"✅ saved {len(df_jumia_products):,} products")

📥 downloading Jumia CSV (~15MB)...
✅ saved 45,338 reviews
📥 downloading Idowenst product catalog from HF...


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.37M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18360 [00:00<?, ? examples/s]

✅ saved 18,360 products


## 5. Pipeline Constants — Personas, Registers, Categories

In [ ]:
PERSONAS = ["chinwe_owerri", "tunde_lagos", "aisha_kano", "femi_abuja", "ifeoma_ph"]
REGISTERS = [
    "nigerian_pidgin", "nigerian_pidgin",          # weighted up
    "code_mixed", "code_mixed",                    # weighted up
    "nigerian_english",
    "standard_english",
]
REGISTERS_SYNTHETIC = [
    "nigerian_pidgin", "nigerian_pidgin",
    "code_mixed", "code_mixed", "code_mixed",      # over-sample in synthetic
    "nigerian_english",
    "standard_english",
]
CATEGORIES = [
    "phone_tablet", "electronics", "appliances", "computing",
    "health_beauty", "fashion", "home_office", "baby_products",
    "sports_outdoor", "automobile",
]
RATINGS = ["1", "2", "3", "4", "5"]

PERSONA_GUIDE = (
    "PERSONAS:\n"
    "chinwe_owerri: Owerri Gen-Z university student, Igbo+English code-mix (nna, biko, ahn ahn), communal-hedonic.\n"
    "tunde_lagos: Lagos market trader 40s, Pidgin-heavy, utilitarian, focused on value/durability.\n"
    "aisha_kano: Kano teacher, measured Nigerian English with Muslim framing (Alhamdulillah, wallahi).\n"
    "femi_abuja: Abuja banker, standard Nigerian English, low-intensity, individualist.\n"
    "ifeoma_ph: PH fashion entrepreneur + Nollywood fan, vibrant Nigerian English with film vocab."
)
REGISTER_GUIDE = (
    "REGISTERS:\n"
    "nigerian_pidgin: Use Pidgin markers naturally: 'abeg', 'no cap', 'wahala', 'e shock me', 'scatter', 'dey', 'go'.\n"
    "code_mixed: Mix Nigerian English with Yoruba/Hausa/Igbo (ahn ahn, haba, nna, biko, owambe, wahala).\n"
    "nigerian_english: Nigerian English markers ('well done sir', 'no shaking', 'sharp sharp', 'sef', 'now').\n"
    "standard_english: Clear standard English, no slang, no Pidgin."
)

print("✅ persona / register / category constants loaded")

✅ persona / register / category constants loaded


## 6. Pipeline 1 — Seed-Grounded: Real Jumia review → (persona, rating, register, rewritten review)

In [ ]:

# Cache the Jumia reviews + products into module-scope lists for fast sampling
P1_SOURCE = df_jumia_reviews.to_dict('records')
random.seed(42); random.shuffle(P1_SOURCE)

RATING_SYS = (
    "You calibrate Nigerian product reviewers. Given a Jumia review and its binary sentiment, "
    "output ONLY a single digit 1-5 matching the text's intensity."
)
REWRITE_SYS = (
    "You rewrite real Nigerian product reviews in specific persona voices and registers. "
    "Preserve sentiment and key points. Sound like a real Nigerian customer. "
    "Output ONLY the rewritten review text — no labels, no quotes, no preamble."
)

async def p1_generate_one(idx: int) -> dict:
    src = P1_SOURCE[idx % len(P1_SOURCE)]
    persona = random.choice(PERSONAS)
    register = random.choice(REGISTERS)
    category = random.choice(CATEGORIES)

    # Step 1: refine binary sentiment → 1-5 rating
    rating_prompt = (
        f"Review: {src['review_text']}\n\n"
        f"Binary sentiment: {src['binary_sentiment']} (+1=pos, -1=neg)\n\n"
        "Scale: 1=rubbish/scam, 2=clearly negative, 3=mixed, 4=clearly positive, 5='e too much'/scatter.\n\n"
        "Output ONLY one digit (1, 2, 3, 4, or 5)."
    )
    rating_raw = await call_openai(PRIMARY_MODEL, RATING_SYS, rating_prompt, max_tokens=5, temperature=0)
    m = re.search(r'[1-5]', rating_raw or "")
    rating = int(m.group()) if m else (4 if src['binary_sentiment']==1 else 2)

    # Step 2: rewrite review in persona voice + register
    rewrite_prompt = (
        f"ORIGINAL REVIEW: {src['review_text']}\n\n"
        f"PERSONA: {persona}\n"
        f"REGISTER: {register}\n"
        f"CATEGORY: {category}\n"
        f"TARGET RATING: {rating}/5\n\n"
        "Rewrite the review:\n"
        "- Keep the SAME sentiment and key points\n"
        "- Match the persona's voice + register strictly\n"
        "- 2-5 sentences, sound real (no AI tells)\n\n"
        f"{PERSONA_GUIDE}\n\n{REGISTER_GUIDE}\n\n"
        "OUTPUT THE REVIEW ONLY:"
    )
    review = await call_openai(PRIMARY_MODEL, REWRITE_SYS, rewrite_prompt, max_tokens=400, temperature=0.75)
    return {
        "persona_id": persona,
        "register_tier": register,
        "product_category": category,
        "refined_rating": rating,
        "persona_review": (review or "").strip(),
        "source_review": src['review_text'][:300],
        "source_sentiment": src['binary_sentiment'],
    }

print("✅ Pipeline 1 generator defined")

✅ Pipeline 1 generator defined


In [ ]:
# Pipeline 1 — preview 5 samples
print("🔍 generating 5 P1 preview samples...")
preview_p1 = asyncio.run(asyncio.gather(*[p1_generate_one(i) for i in range(5)]))
for i, p in enumerate(preview_p1):
    print(f"\n=== sample {i+1} ===")
    print(f"  persona: {p['persona_id']} | register: {p['register_tier']} | rating: {p['refined_rating']}/5")
    print(f"  ORIGINAL: {p['source_review'][:150]}")
    print(f"  REWRITE:  {p['persona_review'][:200]}")

🔍 generating 5 P1 preview samples...

=== sample 1 ===
  persona: femi_abuja | register: standard_english | rating: 2/5
  ORIGINAL: I recieved the new battery. It still does not hold much charge. I need to keep it plugged in to charge 2 or 3 x a day. From talkling to other owners o
  REWRITE:  I received the replacement battery, but it still does not hold charge well. I have to plug it in two or three times a day, which is quite inconvenient. From what other Droid Eris owners have said, thi

=== sample 2 ===
  persona: femi_abuja | register: code_mixed | rating: 2/5
  ORIGINAL: i just got this phone and with all the buying i do on amazon i never thought that i would get a bad product. this is not picking up my sim card and is
  REWRITE:  I just received this phone and honestly, with how often I buy on Amazon, I did not expect this kind of wahala. Haba, it is not detecting my SIM card at all, so I cannot even use it for what I need. I 

=== sample 3 ===
  persona: aisha_kano | register:

In [ ]:
# Override to use official OpenAI API instead of freemodel
LLM_BACKEND = "openai"
OPENAI_BASE_URL = None
WIRE_API = "chat"
PRIMARY_MODEL = "gpt-5.5"  # Change to "gpt-4o" if you want the highest quality and have budget
JUDGE_MODEL = "gpt-5.5"
CONCURRENCY = 20  # Lower concurrency for standard OpenAI tier

# Make sure your official OpenAI API key is set
import os
if os.environ.get("OPENAI_API_KEY") == "lm-studio" or not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("Official OPENAI_API_KEY (sk-...): ")

# Re-initialize the OpenAI client
from openai import AsyncOpenAI
_client_kwargs = {"api_key": os.environ.get("OPENAI_API_KEY"), "timeout": 120.0, "max_retries": 0}
client = AsyncOpenAI(**_client_kwargs)

print(f"✅ Switched to official {LLM_BACKEND} using model {PRIMARY_MODEL}.")
print("You can now re-run the Pipeline 1 cell above.")

✅ Switched to official openai using model gpt-5.5.
You can now re-run the Pipeline 1 cell above.


In [ ]:
# Override to switch back to freemodel API
LLM_BACKEND = "freemodel"
OPENAI_BASE_URL = "https://api.freemodel.dev/v1"
WIRE_API = "responses"
PRIMARY_MODEL = "gpt-5.5"
JUDGE_MODEL = "gpt-5.5"
CONCURRENCY = 40

import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

if not os.environ.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY").startswith("sk-"):
    from getpass import getpass
    print("Please enter your freemodel key (if different from the OpenAI key):")
    os.environ["OPENAI_API_KEY"] = getpass("freemodel API key: ")

# Re-initialize the OpenAI client
from openai import AsyncOpenAI
_client_kwargs = {
    "api_key": os.environ.get("OPENAI_API_KEY"),
    "timeout": 120.0,
    "max_retries": 0,
    "base_url": OPENAI_BASE_URL
}
client = AsyncOpenAI(**_client_kwargs)

print(f"✅ Switched back to {LLM_BACKEND} using model {PRIMARY_MODEL} via {WIRE_API} API.")
print("You can now re-run the Pipeline 1 cell above.")

Please enter your freemodel key (if different from the OpenAI key):


KeyboardInterrupt: Interrupted by user

In [ ]:
# Full Pipeline 1 — batched + resumable
df_p1 = asyncio.run(generate_batched(
    generate_one_fn=p1_generate_one,
    total_records=NUM_P1_RECORDS,
    batch_size=BATCH_SIZE,
    ckpt_dir=P1_CKPT_DIR,
    dataset_name="naija_p1_seed",
    concurrency=CONCURRENCY,
))
print(f"\n✅ Pipeline 1: {len(df_p1):,} records")
if len(df_p1) > 0:
    df_p1.to_parquet(AUGMENTED_DIR / "pipeline1_seed_grounded.parquet", index=False)

🔄 naija_p1_seed: resuming from batch 12 (5,249 done, 6,751 to go)

📦 batch 12 — 500 records (43.7% done)
  ✅ 500 records in 192s (156/min) → batch_012.parquet
  ⏱️ ETA: ~40 min

📦 batch 13 — 500 records (47.9% done)
  ✅ 500 records in 165s (182/min) → batch_013.parquet
  ⏱️ ETA: ~34 min

📦 batch 14 — 500 records (52.1% done)
  ⚠️ 182 errors in this batch — keeping the 318 good ones
      • JSONDecodeError: Expecting value: line 1 column 1 (char 0)
      • HTTPStatusError: Client error '402 Payment Required' for url 'https://api.freemodel.dev/v1/responses'
For more informati
      • HTTPStatusError: Client error '402 Payment Required' for url 'https://api.freemodel.dev/v1/responses'
For more informati
  ✅ 318 records in 110s (173/min) → batch_014.parquet
  ⏱️ ETA: ~32 min

📦 batch 15 — 500 records (54.7% done)
  ⚠️ 500 errors in this batch — keeping the 0 good ones
      • HTTPStatusError: Client error '402 Payment Required' for url 'https://api.freemodel.dev/v1/responses'
For more info

KeyboardInterrupt: 

## 7. Pipeline 2 — Pure Synthetic: persona × product × register × stratified rating

In [ ]:
P2_SOURCE = df_jumia_products.to_dict('records')
random.shuffle(P2_SOURCE)

SYNTH_SYS = (
    "You generate authentic Nigerian Jumia product reviews matching the user's persona voice, "
    "register tier, and target rating. Sound like a real Nigerian customer. "
    "Output ONLY the review text — no labels, no quotes, no preamble."
)

async def p2_generate_one(idx: int) -> dict:
    product = P2_SOURCE[idx % len(P2_SOURCE)]
    persona = random.choice(PERSONAS)
    register = random.choice(REGISTERS_SYNTHETIC)
    rating = random.choice(RATINGS)

    prompt = (
        f"PRODUCT (Jumia):\n"
        f"Title: {product.get('title','')}\n"
        f"Category: {product.get('category','')}\n"
        f"Description: {(product.get('description') or '')[:300]}\n"
        f"Price: ₦{product.get('price_naira','?')}\n\n"
        f"PERSONA: {persona}\n"
        f"REGISTER: {register}\n"
        f"TARGET RATING: {rating}/5\n\n"
        f"Write a {rating}/5 review (2-5 sentences) as this persona. "
        "Don't state the number — let intensity show through tone.\n\n"
        f"{PERSONA_GUIDE}\n\n{REGISTER_GUIDE}\n\n"
        "OUTPUT THE REVIEW ONLY:"
    )
    review = await call_openai(PRIMARY_MODEL, SYNTH_SYS, prompt, max_tokens=400, temperature=0.8)
    return {
        "persona_id": persona,
        "register_tier": register,
        "target_rating": int(rating),
        "title": product.get('title',''),
        "category": product.get('category',''),
        "persona_review": (review or "").strip(),
    }

print("✅ Pipeline 2 generator defined")

In [ ]:
# Pipeline 2 preview
print("🔍 generating 5 P2 preview samples...")
preview_p2 = asyncio.run(asyncio.gather(*[p2_generate_one(i) for i in range(5)]))
for i, p in enumerate(preview_p2):
    print(f"\n=== sample {i+1} ===")
    print(f"  persona: {p['persona_id']} | register: {p['register_tier']} | rating: {p['target_rating']}/5")
    print(f"  product: {p['title'][:80]}")
    print(f"  review:  {p['persona_review'][:200]}")

In [ ]:
# Full Pipeline 2 — batched + resumable
df_p2 = asyncio.run(generate_batched(
    generate_one_fn=p2_generate_one,
    total_records=NUM_P2_RECORDS,
    batch_size=BATCH_SIZE,
    ckpt_dir=P2_CKPT_DIR,
    dataset_name="naija_p2_synthetic",
    concurrency=CONCURRENCY,
))
print(f"\n✅ Pipeline 2: {len(df_p2):,} records")
if len(df_p2) > 0:
    df_p2.to_parquet(AUGMENTED_DIR / "pipeline2_synthetic.parquet", index=False)

In [ ]:
# Load saved data from checkpoints into memory if previous cells were interrupted
df_p1 = load_all_checkpoints(P1_CKPT_DIR)
df_p2 = load_all_checkpoints(P2_CKPT_DIR)

print(f"Loaded {len(df_p1)} Pipeline 1 records.")
print(f"Loaded {len(df_p2)} Pipeline 2 records.")

# Now you can re-run the "Combine P1 + P2" cell below!

  📦 loaded 6,567 from 14 checkpoints
Loaded 6567 Pipeline 1 records.
Loaded 0 Pipeline 2 records.


## 8. Pipeline 3 — LLM-as-Judge Quality Scoring

In [ ]:
# Combine P1 + P2 into a unified format for judging
all_generated = []
if 'df_p1' in dir() and df_p1 is not None and len(df_p1) > 0:
    p1 = df_p1[['persona_review', 'persona_id', 'register_tier', 'product_category', 'refined_rating']].copy()
    p1.columns = ['review', 'persona_id', 'register_tier', 'product_category', 'rating']
    p1['rating'] = pd.to_numeric(p1['rating'], errors='coerce').clip(1,5).fillna(3).astype(int)
    p1['product_title'] = ''
    p1['pipeline'] = 'seed_grounded'
    all_generated.append(p1)
if 'df_p2' in dir() and df_p2 is not None and len(df_p2) > 0:
    p2 = df_p2[['persona_review', 'persona_id', 'register_tier', 'category', 'target_rating', 'title']].copy()
    p2.columns = ['review', 'persona_id', 'register_tier', 'product_category', 'rating', 'product_title']
    p2['rating'] = pd.to_numeric(p2['rating'], errors='coerce').clip(1,5).fillna(3).astype(int)
    p2['pipeline'] = 'synthetic'
    all_generated.append(p2)

df_all = pd.concat(all_generated, ignore_index=True) if all_generated else pd.DataFrame()

if not df_all.empty:
    df_all = df_all.dropna(subset=['review'])
    df_all = df_all[df_all['review'].str.len().between(50, 1500)]
    df_all = df_all.drop_duplicates(subset=['review'], keep='first').reset_index(drop=True)

    print(f"📊 Combined corpus: {len(df_all):,}")
    print(f"   by pipeline:  {dict(df_all['pipeline'].value_counts())}")
    print(f"   by register:  {dict(df_all['register_tier'].value_counts())}")
    print(f"   by rating:    {dict(sorted(df_all['rating'].value_counts().items()))}")
else:
    print("📊 Combined corpus is empty. No data to process.")

df_all.to_parquet(AUGMENTED_DIR / "combined_pre_judge.parquet", index=False)

📊 Combined corpus: 6,525
   by pipeline:  {'seed_grounded': np.int64(6525)}
   by register:  {'code_mixed': np.int64(2240), 'nigerian_pidgin': np.int64(2110), 'standard_english': np.int64(1142), 'nigerian_english': np.int64(1033)}
   by rating:    {1: 733, 2: 1435, 3: 456, 4: 2811, 5: 1090}


In [ ]:
# Judge generator — scores 4 dimensions on 0-4 scale via JSON output
JUDGE_SYS = (
    "You evaluate Nigerian product reviews on 4 dimensions: register_fidelity, persona_authenticity, "
    "rating_coherence, naturalness. Return STRICT JSON with integer scores 0-4 for each. "
    "4=perfect, 3=good, 2=partial, 1=weak, 0=bad."
)
JUDGE_PROMPT_TPL = (
    "Evaluate this Nigerian product review:\n\n"
    "PERSONA: {persona_id}\n"
    "DECLARED REGISTER: {register_tier}\n"
    "TARGET RATING: {rating}/5\n"
    "REVIEW: {review}\n\n"
    "Score 0-4 on each dimension:\n"
    "- register_fidelity: does language match the declared register tier?\n"
    "- persona_authenticity: sounds like the declared persona archetype?\n"
    "- rating_coherence: text intensity matches the target rating?\n"
    "- naturalness: reads as real Nigerian (not AI)?\n\n"
    'Return STRICT JSON: {{"register_fidelity": int, "persona_authenticity": int, '
    '"rating_coherence": int, "naturalness": int}}'
)

JUDGE_N = min(JUDGE_SAMPLE, len(df_all))
JUDGE_INPUT = df_all.sample(n=JUDGE_N, random_state=42).reset_index(drop=True).to_dict('records')

async def judge_generate_one(idx: int) -> dict:
    src = JUDGE_INPUT[idx]
    prompt = JUDGE_PROMPT_TPL.format(
        persona_id=src['persona_id'],
        register_tier=src['register_tier'],
        rating=src['rating'],
        review=src['review'][:1200],
    )
    raw = await call_openai(
        JUDGE_MODEL, JUDGE_SYS, prompt,
        max_tokens=200, temperature=0.1,
        response_format={"type": "json_object"},
    )
    try:
        scores = json.loads(raw)
    except Exception:
        scores = {}
    return {
        "review": src['review'],
        "persona_id": src['persona_id'],
        "register_tier": src['register_tier'],
        "rating": src['rating'],
        "register_fidelity":     int(scores.get('register_fidelity', 0) or 0),
        "persona_authenticity":  int(scores.get('persona_authenticity', 0) or 0),
        "rating_coherence":      int(scores.get('rating_coherence', 0) or 0),
        "naturalness":           int(scores.get('naturalness', 0) or 0),
    }

df_judged = asyncio.run(generate_batched(
    generate_one_fn=judge_generate_one,
    total_records=JUDGE_N,
    batch_size=BATCH_SIZE,
    ckpt_dir=P3_CKPT_DIR,
    dataset_name="naija_p3_judge",
    concurrency=CONCURRENCY,
))

score_cols = ['register_fidelity', 'persona_authenticity', 'rating_coherence', 'naturalness']
for c in score_cols:
    df_judged[c] = pd.to_numeric(df_judged[c], errors='coerce')
df_judged['composite_score'] = df_judged[score_cols].mean(axis=1)

print("\n📊 QUALITY SCORES")
for c in score_cols:
    print(f"   {c:25s}  mean={df_judged[c].mean():.2f}  std={df_judged[c].std():.2f}")
print(f"   {'composite':25s}  mean={df_judged['composite_score'].mean():.2f}")
hq = df_judged[df_judged['composite_score'] >= 3.0]
print(f"\n   high-quality (≥3.0): {len(hq):,}/{len(df_judged):,} ({100*len(hq)/max(len(df_judged),1):.1f}%)")
df_judged.to_parquet(AUGMENTED_DIR / "quality_scored_sample.parquet", index=False)

🚀 naija_p3_judge: starting fresh (200 target, concurrency=20)

📦 batch 1 — 200 records (0.0% done)
  ⚠️ 2 errors in this batch — keeping the 198 good ones
      • BadRequestError: Error code: 400 - {'error': {'message': 'Could not finish the message because max_tokens or model outpu
      • BadRequestError: Error code: 400 - {'error': {'message': 'Could not finish the message because max_tokens or model outpu
  ✅ 198 records in 45s (263/min) → batch_001.parquet
  ⏱️ ETA: ~0 min

📦 batch 2 — 2 records (99.0% done)
  ✅ 2 records in 3s (39/min) → batch_002.parquet
  ⏱️ ETA: ~0 min

✅ session in 0.8min
  📦 loaded 200 from 2 checkpoints

📊 QUALITY SCORES
   register_fidelity          mean=3.29  std=1.34
   persona_authenticity       mean=2.82  std=1.31
   rating_coherence           mean=3.40  std=1.35
   naturalness                mean=2.93  std=1.27
   composite                  mean=3.11

   high-quality (≥3.0): 171/200 (85.5%)


## 9. Final Filter + Split + Export to Drive

In [ ]:
df_final = df_all.copy()
if 'composite_score' in df_judged.columns and len(df_judged):
    score_map = df_judged.set_index('review')['composite_score'].to_dict()
    df_final['composite_score'] = df_final['review'].map(score_map)
    n_before = len(df_final)
    keep = (df_final['composite_score'].isna()) | (df_final['composite_score'] >= 3.0)
    df_final = df_final[keep].reset_index(drop=True)
    print(f"🔍 judge filter: {n_before:,} → {len(df_final):,}")

# Stratified 90/5/5 split by register
_r = random.Random(42)
buckets = defaultdict(list)
for _, row in df_final.iterrows():
    buckets[row['register_tier']].append(row.to_dict())
train, val, test = [], [], []
for tier, items in buckets.items():
    _r.shuffle(items)
    n = len(items); n_v = max(1, n//20); n_t = max(1, n//20)
    val   += items[:n_v]
    test  += items[n_v:n_v+n_t]
    train += items[n_v+n_t:]
_r.shuffle(train); _r.shuffle(val); _r.shuffle(test)
print(f"📊 splits: train={len(train):,}  val={len(val):,}  test={len(test):,}")

INSTRUCTION = (
    "Simulate the review behaviour of the following Nigerian user reviewing the described "
    "product. Generate the rating (1-5) and review text exactly as this user would write it. "
    "Match the user's register tier and cultural framing."
)

def to_alpaca(r):
    persona = json.dumps({"persona_id": r['persona_id'], "register_tier": r['register_tier']}, ensure_ascii=False)
    product = json.dumps({"title": r.get('product_title',''), "category": r.get('product_category','')}, ensure_ascii=False)
    output  = json.dumps({"rating": int(r['rating']), "review": r['review']}, ensure_ascii=False)
    return {
        "instruction": INSTRUCTION,
        "input": f"### Persona\n{persona}\n\n### Product\n{product}\n\n### Register tier\n{r['register_tier']}",
        "output": output,
    }

def to_sharegpt(r):
    persona = json.dumps({"persona_id": r['persona_id'], "register_tier": r['register_tier']}, ensure_ascii=False)
    product = json.dumps({"title": r.get('product_title',''), "category": r.get('product_category','')}, ensure_ascii=False)
    user = f"{INSTRUCTION}\n\n### Persona\n{persona}\n\n### Product\n{product}\n\n### Register tier\n{r['register_tier']}"
    asst = json.dumps({"rating": int(r['rating']), "review": r['review']}, ensure_ascii=False)
    return {"conversations": [{"from":"human","value":user}, {"from":"gpt","value":asst}]}

for name, rows in [("train", train), ("val", val), ("test", test)]:
    with jsonlines.open(PROCESSED_DIR / f"v1_{name}_alpaca.jsonl", mode='w') as w:
        for r in rows: w.write(to_alpaca(r))
    with jsonlines.open(PROCESSED_DIR / f"v1_{name}_sharegpt.jsonl", mode='w') as w:
        for r in rows: w.write(to_sharegpt(r))
    pd.DataFrame(rows).to_parquet(PROCESSED_DIR / f"v1_{name}_full.parquet", index=False)
    print(f"  💾 {name}: {len(rows):,} → alpaca + sharegpt + parquet")

# Combined full corpus (no split)
all_rows = train + val + test
with jsonlines.open(PROCESSED_DIR / "final_alpaca_format.jsonl", mode='w') as w:
    for r in all_rows: w.write(to_alpaca(r))
with jsonlines.open(PROCESSED_DIR / "final_sharegpt_format.jsonl", mode='w') as w:
    for r in all_rows: w.write(to_sharegpt(r))
pd.DataFrame(all_rows).to_parquet(PROCESSED_DIR / "final_full_dataset.parquet", index=False)

# Statistics
final_stats = {
    "total_pairs": len(all_rows),
    "splits": {"train": len(train), "val": len(val), "test": len(test)},
    "by_pipeline": dict(pd.DataFrame(all_rows)['pipeline'].value_counts()),
    "by_register": dict(pd.DataFrame(all_rows)['register_tier'].value_counts()),
    "by_rating":   dict(sorted(pd.DataFrame(all_rows)['rating'].value_counts().items())),
    "generated_at": datetime.now().isoformat(),
    "generated_with": f"OpenAI {PRIMARY_MODEL} + {JUDGE_MODEL} judge",
}
(METADATA_DIR / "final_statistics.json").write_text(json.dumps(final_stats, indent=2, default=str))

print(f"\n{'='*60}")
print("🏆 FINAL STATS")
print(f"{'='*60}")
print(json.dumps(final_stats, indent=2, default=str))

🔍 judge filter: 6,525 → 6,496
📊 splits: train=5,852  val=322  test=322
  💾 train: 5,852 → alpaca + sharegpt + parquet
  💾 val: 322 → alpaca + sharegpt + parquet
  💾 test: 322 → alpaca + sharegpt + parquet

🏆 FINAL STATS
{
  "total_pairs": 6496,
  "splits": {
    "train": 5852,
    "val": 322,
    "test": 322
  },
  "by_pipeline": {
    "seed_grounded": "6496"
  },
  "by_register": {
    "code_mixed": "2230",
    "nigerian_pidgin": "2097",
    "standard_english": "1139",
    "nigerian_english": "1030"
  },
  "by_rating": {
    "1": 729,
    "2": 1428,
    "3": 454,
    "4": 2801,
    "5": 1084
  },
  "generated_at": "2026-05-17T06:21:44.364038",
  "generated_with": "OpenAI gpt-5.5 + gpt-5.5 judge"
}


In [ ]:
# Output directory tree
print("\n📁 Output Directory:")
print("="*60)
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    for f in sorted(files):
        fp = Path(root) / f
        size = fp.stat().st_size / (1024*1024)
        print(f"{indent}  📄 {f} ({size:.2f} MB)")

print(f"""
{'='*70}
✅ NAIJA CORPUS COMPLETE — built via OpenAI {PRIMARY_MODEL}
{'='*70}

Drive location:
   {OUTPUT_DIR}

For fine-tuning (open 02_finetune.ipynb on a GPU runtime):
   processed/v1_train_alpaca.jsonl
   processed/v1_val_alpaca.jsonl
   processed/v1_test_alpaca.jsonl

Next: switch to GPU runtime → open 02_finetune.ipynb
   (the default CORPUS_SOURCE="openai" will pick this corpus).
""")


📁 Output Directory:
📂 naija_persona_corpus_openai/
  📂 raw/
    📄 jumia_products_seed.parquet (4.02 MB)
    📄 jumia_reviews_raw.csv (15.02 MB)
    📄 jumia_reviews_seed.parquet (6.90 MB)
  📂 processed/
    📄 final_alpaca_format.jsonl (4.71 MB)
    📄 final_full_dataset.parquet (0.99 MB)
    📄 final_sharegpt_format.jsonl (4.95 MB)
    📄 v1_test_alpaca.jsonl (0.24 MB)
    📄 v1_test_full.parquet (0.06 MB)
    📄 v1_test_sharegpt.jsonl (0.25 MB)
    📄 v1_train_alpaca.jsonl (4.24 MB)
    📄 v1_train_full.parquet (0.89 MB)
    📄 v1_train_sharegpt.jsonl (4.45 MB)
    📄 v1_val_alpaca.jsonl (0.23 MB)
    📄 v1_val_full.parquet (0.06 MB)
    📄 v1_val_sharegpt.jsonl (0.25 MB)
  📂 metadata/
    📄 final_statistics.json (0.00 MB)
  📂 augmented/
    📄 combined_pre_judge.parquet (0.99 MB)
    📄 quality_scored_sample.parquet (0.04 MB)
    📂 checkpoints/
      📂 pipeline1/
        📄 batch_001.jsonl (0.30 MB)
        📄 batch_001.parquet (0.13 MB)
        📄 batch_002.jsonl (0.30 MB)
        📄 batch_002.parque

## 🔧 Utility — Reset checkpoints (uncomment to regenerate)

In [ ]:
# import shutil
# shutil.rmtree(P1_CKPT_DIR, ignore_errors=True); P1_CKPT_DIR.mkdir(parents=True, exist_ok=True); print("🗑️ P1 wiped")
# shutil.rmtree(P2_CKPT_DIR, ignore_errors=True); P2_CKPT_DIR.mkdir(parents=True, exist_ok=True); print("🗑️ P2 wiped")
# shutil.rmtree(P3_CKPT_DIR, ignore_errors=True); P3_CKPT_DIR.mkdir(parents=True, exist_ok=True); print("🗑️ P3 wiped")